# 單元四：綜合實戰演練 — 公司月報自動化

> **學習目標：**
> - 能整合 ChatGPT + Colab + Gemini CLI 完成完整自動化流程
> - 能依照實際需求選擇最適合的 AI 工具
> - 完成一個從原始資料到自動月報的完整範例

### 情境描述

你是公司的業務助理，每月需要：
1. 收到 5 個部門的 Excel 銷售報表
2. 合併所有資料
3. 製作月報（含圖表）
4. 寄給主管

---

## Step 0：安裝套件 + 上傳 5 個部門檔案

請上傳講師提供的 5 個部門 Excel：
- `業務部.xlsx`
- `行銷部.xlsx`
- `客服部.xlsx`
- `研發部.xlsx`
- `管理部.xlsx`

In [ ]:
!pip install openpyxl pandas xlsxwriter matplotlib

In [ ]:
from google.colab import files

# 上傳 5 個部門的 Excel 檔案
uploaded = files.upload()
print(f'已上傳 {len(uploaded)} 個檔案：')
for name in uploaded:
    print(f'  {name}')

---
## Step 1：合併所有部門資料

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ===== 步驟 1：合併所有部門資料 =====
departments = ['業務部', '行銷部', '客服部', '研發部', '管理部']
all_data = []

for dept in departments:
    try:
        df = pd.read_excel(f'{dept}.xlsx')
        df['部門'] = dept
        all_data.append(df)
        print(f"已讀取：{dept} ({len(df)} 筆)")
    except FileNotFoundError:
        print(f"找不到：{dept}.xlsx，跳過")

combined = pd.concat(all_data, ignore_index=True)
print(f"\n合併完成，共 {len(combined)} 筆資料")
print(f"欄位：{list(combined.columns)}")
combined.head(10)

---
## Step 2：統計分析

In [ ]:
# ===== 步驟 2：統計分析 =====

# 各部門銷售統計
dept_summary = combined.groupby('部門').agg(
    訂單數=('金額', 'count'),
    總營收=('金額', 'sum'),
    平均客單價=('金額', 'mean'),
    最大訂單=('金額', 'max')
).round(0)

# 月度趨勢
combined['日期'] = pd.to_datetime(combined['日期'])
combined['月份'] = combined['日期'].dt.month
monthly = combined.groupby('月份')['金額'].sum()

# Top 10 業務員
top_sales = combined.groupby('業務員')['金額'].sum().nlargest(10)

print("=== 各部門銷售統計 ===")
print(dept_summary)
print("\n=== 月度營收 ===")
print(monthly)
print("\n=== Top 10 業務員 ===")
print(top_sales)

---
## Step 3：生成月報 Excel

In [ ]:
# ===== 步驟 3：生成報表 =====
with pd.ExcelWriter('月報.xlsx', engine='openpyxl') as writer:
    dept_summary.to_excel(writer, sheet_name='部門摘要')
    top_sales.to_excel(writer, sheet_name='業務員排行')
    combined.to_excel(writer, sheet_name='完整明細', index=False)

print("月報已生成：月報.xlsx")

# 下載月報
from google.colab import files
files.download('月報.xlsx')

---
## Step 4：生成圖表

In [ ]:
import matplotlib.pyplot as plt

# 設定中文字體
plt.rcParams['font.sans-serif'] = ['Noto Sans CJK TC', 'Microsoft JhengHei', 'SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ===== 步驟 4：生成圖表 =====
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 各部門營收
dept_summary['總營收'].plot(kind='bar', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('各部門營收')
axes[0,0].set_ylabel('金額 (NT$)')

# 月度趨勢
monthly.plot(kind='line', ax=axes[0,1], marker='o', color='green')
axes[0,1].set_title('月度營收趨勢')
axes[0,1].set_xlabel('月份')
axes[0,1].set_ylabel('金額 (NT$)')

# Top 10 業務員
top_sales.plot(kind='barh', ax=axes[1,0], color='coral')
axes[1,0].set_title('Top 10 業務員')
axes[1,0].set_xlabel('金額 (NT$)')

# 各部門占比
dept_summary['總營收'].plot(kind='pie', ax=axes[1,1], autopct='%1.1f%%')
axes[1,1].set_title('營收占比')
axes[1,1].set_ylabel('')

plt.tight_layout()
plt.savefig('月報圖表.png', dpi=150)
plt.show()
print("圖表已儲存：月報圖表.png")

# 下載圖表
from google.colab import files
files.download('月報圖表.png')

---
## Step 5（加分題）：用 Colab 內建 AI 撰寫月報摘要

**方法 A：使用 Colab AI 面板**

在 Colab 右上角 AI 面板輸入：
```
以下是本月銷售數據摘要，請用繁體中文撰寫一份簡短的月報摘要（200 字以內），
包含：本月整體表現、表現最好的部門、需要注意的地方、下月建議。
```
然後把上面 Step 2 的統計結果貼給它。

**方法 B：用程式呼叫 Gemini API**

In [ ]:
# 如果要用程式自動產生月報摘要，可使用 Gemini API
# 請先執行：!pip install google-generativeai
# 並填入你的 API Key

# import google.generativeai as genai
# genai.configure(api_key='你的_API_KEY')
#
# summary_text = f"""
# 各部門銷售統計：
# {dept_summary.to_string()}
#
# Top 10 業務員：
# {top_sales.to_string()}
# """
#
# model = genai.GenerativeModel('gemini-2.0-flash')
# response = model.generate_content(
#     f"請用繁體中文撰寫一份簡短的月報摘要（200 字以內），"
#     f"包含：本月整體表現、最佳部門、注意事項、下月建議。\n{summary_text}"
# )
# print("=== AI 月報摘要 ===")
# print(response.text)

---
## 課堂挑戰

> 試著修改上面的程式，完成以下進階任務：
> 1. 在月報 Excel 中加上「各產品類別銷售統計」工作表
> 2. 新增一張折線圖顯示「各部門月度趨勢」
> 3. 用 Colab AI 面板協助你完成修改

### 自行練習區

In [ ]:
# 請在這裡撰寫你的練習程式

